In [1]:
"""
Assignment 1 – NLP-Based IT Support Chatbot
Module: CONL725 – Artificial Intelligence in Theory and Practice

Final version prepared for submission.

What this script includes:
1. Data collection (labeled IT-support dataset)
2. Text preprocessing (tokenization, lowercasing, stopword removal, lemmatization)
3. Feature extraction using Bag-of-Words
4. Naive Bayes model training
5. Model evaluation using accuracy
6. Test cases with visible results
7. Interactive chatbot with confidence threshold
"""

# =========================
# 1. IMPORTS
# =========================
import random
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# =========================
# 2. NLTK DOWNLOADS
# =========================
# These are required the first time the code runs.
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# =========================
# 3. DATASET
# =========================
# Each tuple contains: (user_query, intent_label)
# Intents used:
# - Password
# - Connectivity
# - Email
# - Printer
# - LMS

dataset = [
    # PASSWORD
    ("I forgot my password", "Password"),
    ("How do I reset my password?", "Password"),
    ("My university password is not working", "Password"),
    ("I need to change my password", "Password"),
    ("Password reset link not working", "Password"),
    ("Cannot login because of password error", "Password"),
    ("Forgot my portal password", "Password"),
    ("My account password expired", "Password"),
    ("Password is incorrect", "Password"),
    ("Unable to update my password", "Password"),
    ("System says password invalid", "Password"),
    ("Need help resetting password", "Password"),
    ("How can I recover my password?", "Password"),
    ("Login failed due to wrong password", "Password"),
    ("Password not accepted", "Password"),
    ("I can't remember my password", "Password"),
    ("Change account password", "Password"),
    ("Password recovery issue", "Password"),
    ("Forgot email password", "Password"),
    ("Password problem", "Password"),

    # CONNECTIVITY
    ("WiFi is not working", "Connectivity"),
    ("Internet connection is very slow", "Connectivity"),
    ("I cannot connect to the network", "Connectivity"),
    ("VPN is disconnected", "Connectivity"),
    ("No internet access in my office", "Connectivity"),
    ("Campus WiFi keeps dropping", "Connectivity"),
    ("My laptop cannot access the internet", "Connectivity"),
    ("Connection timed out", "Connectivity"),
    ("Unable to connect to WiFi", "Connectivity"),
    ("The network is unavailable", "Connectivity"),
    ("Internet is unstable today", "Connectivity"),
    ("VPN login failed", "Connectivity"),
    ("Why is the internet not working?", "Connectivity"),
    ("Cannot access websites", "Connectivity"),
    ("My internet stopped suddenly", "Connectivity"),
    ("Wireless connection problem", "Connectivity"),
    ("I lost network connection", "Connectivity"),
    ("Internet issue on campus", "Connectivity"),
    ("Unable to join the network", "Connectivity"),
    ("Connection issue", "Connectivity"),

    # EMAIL
    ("Outlook is not opening", "Email"),
    ("I am not receiving emails", "Email"),
    ("Cannot send email", "Email"),
    ("My email account is not syncing", "Email"),
    ("Outlook keeps crashing", "Email"),
    ("Email password is wrong", "Email"),
    ("Mailbox is full", "Email"),
    ("I cannot login to my email", "Email"),
    ("Why is Outlook offline?", "Email"),
    ("Email not updating", "Email"),
    ("Missing emails in inbox", "Email"),
    ("Cannot access staff email", "Email"),
    ("Outlook shows an error", "Email"),
    ("Email configuration problem", "Email"),
    ("I did not get the message", "Email"),
    ("Unable to send attachments", "Email"),
    ("My email is not working", "Email"),
    ("Outlook login issue", "Email"),
    ("Exchange mail problem", "Email"),
    ("Email issue", "Email"),

    # PRINTER
    ("Printer is offline", "Printer"),
    ("I cannot print my document", "Printer"),
    ("Printer is not responding", "Printer"),
    ("Paper jam in the printer", "Printer"),
    ("Printer driver issue", "Printer"),
    ("The printer is out of ink", "Printer"),
    ("Printer queue is stuck", "Printer"),
    ("Unable to connect to printer", "Printer"),
    ("Shared printer not found", "Printer"),
    ("Printer is printing blank pages", "Printer"),
    ("Cannot scan from printer", "Printer"),
    ("Printer error on screen", "Printer"),
    ("My printer is disconnected", "Printer"),
    ("Printer not available", "Printer"),
    ("Printing is very slow", "Printer"),
    ("Office printer problem", "Printer"),
    ("Need help with printer setup", "Printer"),
    ("Cannot find the network printer", "Printer"),
    ("Printer issue", "Printer"),
    ("The document is not printing", "Printer"),

    # LMS
    ("I cannot log in to the LMS", "LMS"),
    ("The LMS page is not loading", "LMS"),
    ("Course materials are missing", "LMS"),
    ("I cannot access my course", "LMS"),
    ("LMS login failed", "LMS"),
    ("The quiz is not opening", "LMS"),
    ("I cannot submit my assignment", "LMS"),
    ("The LMS is very slow", "LMS"),
    ("My grades are not visible", "LMS"),
    ("The discussion board is not working", "LMS"),
    ("LMS error message", "LMS"),
    ("Unable to join course", "LMS"),
    ("I cannot upload files to LMS", "LMS"),
    ("LMS dashboard problem", "LMS"),
    ("My course disappeared", "LMS"),
    ("Cannot open lecture slides", "LMS"),
    ("Assignment upload failed", "LMS"),
    ("LMS access issue", "LMS"),
    ("The portal class page is broken", "LMS"),
    ("Learning system problem", "LMS"),
]


# =========================
# 4. PREPROCESSING
# =========================
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess(text):
    """
    Convert raw input text into cleaned tokens.

    Steps:
    1. Lowercase
    2. Tokenize
    3. Keep alphabetic words only
    4. Lemmatize
    5. Remove stopwords
    """
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    tokens = [word for word in tokens if word not in stop_words]
    return tokens


# =========================
# 5. PREPARE DATASET
# =========================
processed_dataset = [(preprocess(text), label) for text, label in dataset]


# =========================
# 6. FEATURE EXTRACTION
# =========================
# Build vocabulary from all training words.
all_words = sorted({word for tokens, label in processed_dataset for word in tokens})


def extract_features(tokens):
    """
    Convert token list into Bag-of-Words features.
    Each vocabulary word becomes True/False.
    """
    token_set = set(tokens)
    return {word: (word in token_set) for word in all_words}


featuresets = [(extract_features(tokens), label) for tokens, label in processed_dataset]


# =========================
# 7. TRAIN / TEST SPLIT
# =========================
random.seed(42)
random.shuffle(featuresets)

train_size = int(len(featuresets) * 0.8)
train_set = featuresets[:train_size]
test_set = featuresets[train_size:]


# =========================
# 8. TRAIN MODEL
# =========================
classifier = nltk.NaiveBayesClassifier.train(train_set)


# =========================
# 9. EVALUATION
# =========================
accuracy = nltk.classify.accuracy(classifier, test_set)
print("\n=== MODEL EVALUATION ===")
print(f"Model Accuracy: {accuracy:.2%}")


# =========================
# 10. INFORMATIVE FEATURES
# =========================
print("\n=== MOST INFORMATIVE FEATURES ===")
classifier.show_most_informative_features(10)


# =========================
# 11. TEST CASES AND RESULTS
# =========================
# These examples show that the model can classify sample queries.
test_sentences = [
    "I forgot my password",
    "The printer is offline",
    "Outlook is not receiving emails",
    "I cannot access my course on LMS",
    "The WiFi keeps disconnecting",
    "My browser is broken",  # unknown / weak example
]

print("\n=== TEST RESULTS ===")
for sentence in test_sentences:
    tokens = preprocess(sentence)
    features = extract_features(tokens)
    predicted = classifier.classify(features)
    confidence = classifier.prob_classify(features).prob(predicted)

    print(f"Input: {sentence}")
    print(f"Clean Tokens: {tokens}")
    print(f"Predicted Intent: {predicted}")
    print(f"Confidence: {confidence:.2f}")
    print("-" * 50)


# =========================
# 12. RESPONSE DICTIONARY
# =========================
responses = {
    "Password": (
        "To reset your password, please visit the university portal and click "
        "'Forgot Password'. If the issue persists, contact IT support."
    ),
    "Connectivity": (
        "Please check your WiFi or VPN connection. Try restarting your router "
        "or reconnecting to the campus network."
    ),
    "Email": (
        "Please restart Outlook or check your email configuration settings. "
        "Ensure your account credentials are correct."
    ),
    "Printer": (
        "Please verify that the printer is powered on and connected to the "
        "network. Try reinstalling the printer driver if needed."
    ),
    "LMS": (
        "Please clear your browser cache and try logging into the LMS again. "
        "If the issue continues, contact academic IT support."
    ),
}


# =========================
# 13. CHATBOT FUNCTION
# =========================
def chatbot(confidence_threshold=0.50):
    """Run the interactive IT support chatbot."""
    print("\nIT Support Chatbot (type 'exit' to quit)")
    print("-" * 50)

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == "exit":
            print("Chatbot: Thank you for using IT Support. Goodbye!")
            break

        tokens = preprocess(user_input)
        features = extract_features(tokens)

        predicted_intent = classifier.classify(features)
        probability_distribution = classifier.prob_classify(features)
        confidence = probability_distribution.prob(predicted_intent)

        if confidence < confidence_threshold:
            print(
                f"Chatbot ({predicted_intent}, confidence: {confidence:.2f}): "
                "Sorry, I am not fully sure about your issue. "
                "Please contact IT support for further assistance."
            )
        else:
            response = responses.get(
                predicted_intent,
                "Please contact IT support for further assistance.",
            )
            print(
                f"Chatbot ({predicted_intent}, confidence: {confidence:.2f}): "
                f"{response}"
            )


# =========================
# 14. PROGRAM ENTRY POINT
# =========================
if __name__ == "__main__":
    chatbot()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



=== MODEL EVALUATION ===
Model Accuracy: 85.00%

=== MOST INFORMATIVE FEATURES ===
Most Informative Features
                password = True           Passwo : Email  =     12.9 : 1.0
                   email = True            Email : Passwo =      5.7 : 1.0
                 printer = False           Email : Printe =      5.0 : 1.0
                  access = True           Connec : Email  =      2.9 : 1.0
                 network = True           Connec : Printe =      2.6 : 1.0
                   email = False          Printe : Email  =      2.3 : 1.0
                 account = True           Passwo : Email  =      2.2 : 1.0
                 working = True           Connec : Email  =      2.1 : 1.0
                   issue = True           Connec : Printe =      1.9 : 1.0
                internet = False           Email : Connec =      1.8 : 1.0

=== TEST RESULTS ===
Input: I forgot my password
Clean Tokens: ['forgot', 'password']
Predicted Intent: Password
Confidence: 0.99
---------

You:  exit


Chatbot: Thank you for using IT Support. Goodbye!


In [2]:
-------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 3
      1 sample_text = "I forgot my password!!!"
      2 print("Input:", sample_text)
----> 3 print("Output:", preprocess(sample_text))

NameError: name 'preprocess' is not defined

SyntaxError: invalid syntax (3830051844.py, line 1)

In [3]:
sample_text = "I forgot my password!!!"
print("Input:", sample_text)
print("Output:", preprocess(sample_text))

Input: I forgot my password!!!


NameError: name 'preprocess' is not defined

In [4]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [w for w in tokens if w.isalpha()]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
sample_text = "I forgot my password!!!"
print("Input:", sample_text)
print("Output:", preprocess(sample_text))

Input: I forgot my password!!!
Output: ['forgot', 'password']


In [6]:
print("=== MODEL EVALUATION ===")
print(f"Model Accuracy: {accuracy:.2%}")

=== MODEL EVALUATION ===


NameError: name 'accuracy' is not defined

In [7]:
import random
import nltk
from nltk.classify import accuracy as nltk_accuracy

# تجهيز البيانات
processed_dataset = [(preprocess(text), label) for text, label in dataset]

all_words = sorted({word for tokens, label in processed_dataset for word in tokens})

def extract_features(tokens):
    return {word: (word in tokens) for word in all_words}

featuresets = [(extract_features(tokens), label) for tokens, label in processed_dataset]

random.shuffle(featuresets)

train_size = int(len(featuresets) * 0.8)
train_set = featuresets[:train_size]
test_set = featuresets[train_size:]

# تدريب الموديل
classifier = nltk.NaiveBayesClassifier.train(train_set)

# حساب accuracy
accuracy = nltk_accuracy(classifier, test_set)

NameError: name 'dataset' is not defined

In [8]:
print("=== MODEL EVALUATION ===")
print(f"Model Accuracy: {accuracy:.2%}")

=== MODEL EVALUATION ===


NameError: name 'accuracy' is not defined

In [9]:
dataset = [
    ("I forgot my password", "Password"),
    ("The printer is offline", "Printer"),
    ("Outlook is not receiving emails", "Email"),
    ("I cannot access my course", "LMS"),
    ("WiFi is not working", "Connectivity"),
    ("Reset my password", "Password"),
    ("Printer not working", "Printer"),
    ("Email not sending", "Email"),
    ("LMS not loading", "LMS"),
    ("Internet connection issue", "Connectivity"),
]

In [10]:
processed_dataset = [(preprocess(text), label) for text, label in dataset]
...
classifier = nltk.NaiveBayesClassifier.train(train_set)
accuracy = nltk_accuracy(classifier, test_set)

NameError: name 'train_set' is not defined

In [11]:
import random
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.classify import accuracy as nltk_accuracy

# تحميل
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# dataset
dataset = [
    ("I forgot my password", "Password"),
    ("The printer is offline", "Printer"),
    ("Outlook is not receiving emails", "Email"),
    ("I cannot access my course", "LMS"),
    ("WiFi is not working", "Connectivity"),
    ("Reset my password", "Password"),
    ("Printer not working", "Printer"),
    ("Email not sending", "Email"),
    ("LMS not loading", "LMS"),
    ("Internet connection issue", "Connectivity"),
]

# preprocessing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [w for w in tokens if w.isalpha()]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

# تجهيز البيانات
processed_dataset = [(preprocess(text), label) for text, label in dataset]

all_words = sorted({word for tokens, label in processed_dataset for word in tokens})

def extract_features(tokens):
    return {word: (word in tokens) for word in all_words}

featuresets = [(extract_features(tokens), label) for tokens, label in processed_dataset]

# تقسيم
random.shuffle(featuresets)
train_size = int(len(featuresets) * 0.8)
train_set = featuresets[:train_size]
test_set = featuresets[train_size:]

# تدريب
classifier = nltk.NaiveBayesClassifier.train(train_set)

# تقييم
accuracy = nltk_accuracy(classifier, test_set)

print("=== MODEL EVALUATION ===")
print(f"Model Accuracy: {accuracy:.2%}")

=== MODEL EVALUATION ===
Model Accuracy: 50.00%


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alihantash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
accuracy = nltk_accuracy(classifier, test_set)

print("=== MODEL EVALUATION ===")
print(f"Model Accuracy: {accuracy:.2%}")

=== MODEL EVALUATION ===
Model Accuracy: 50.00%


In [13]:
test_sentences = [
    "I forgot my password",
    "The printer is offline",
    "Outlook is not receiving emails",
    "I cannot access my course on LMS",
    "The WiFi keeps disconnecting",
    "My browser is broken"
]

print("=== TEST RESULTS ===")

for sentence in test_sentences:
    tokens = preprocess(sentence)
    features = extract_features(tokens)
    predicted = classifier.classify(features)
    confidence = classifier.prob_classify(features).prob(predicted)

    print(f"Input: {sentence}")
    print(f"Clean Tokens: {tokens}")
    print(f"Predicted Intent: {predicted}")
    print(f"Confidence: {confidence:.2f}")
    print("-" * 50)

=== TEST RESULTS ===
Input: I forgot my password
Clean Tokens: ['forgot', 'password']
Predicted Intent: Password
Confidence: 0.55
--------------------------------------------------
Input: The printer is offline
Clean Tokens: ['printer', 'offline']
Predicted Intent: Printer
Confidence: 0.98
--------------------------------------------------
Input: Outlook is not receiving emails
Clean Tokens: ['outlook', 'receiving', 'email']
Predicted Intent: Email
Confidence: 0.99
--------------------------------------------------
Input: I cannot access my course on LMS
Clean Tokens: ['access', 'course', 'lm']
Predicted Intent: LMS
Confidence: 0.99
--------------------------------------------------
Input: The WiFi keeps disconnecting
Clean Tokens: ['wifi', 'keep', 'disconnecting']
Predicted Intent: LMS
Confidence: 0.48
--------------------------------------------------
Input: My browser is broken
Clean Tokens: ['browser', 'broken']
Predicted Intent: LMS
Confidence: 0.50
-------------------------------

In [16]:
chatbot()

NameError: name 'chatbot' is not defined

In [17]:
def chatbot():
    print("\nIT Support Chatbot (type 'exit' to quit)")
    print("-" * 50)

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Chatbot: Goodbye!")
            break

        tokens = preprocess(user_input)
        features = extract_features(tokens)

        predicted = classifier.classify(features)
        confidence = classifier.prob_classify(features).prob(predicted)

        if confidence < 0.5:
            print("Chatbot: Sorry, I am not sure. Please contact IT support.")
        else:
            print(f"Chatbot: {predicted} issue detected. Please follow IT support instructions.")

In [18]:
chatbot()


IT Support Chatbot (type 'exit' to quit)
--------------------------------------------------


You:  I forgot my password


Chatbot: Password issue detected. Please follow IT support instructions.


You:  exit


Chatbot: Goodbye!


In [1]:
sentences = [
    ("I forgot my password", "Password"),
    ("My internet is slow", "Connectivity"),
    ("Email not working", "Email"),
    ("Printer is broken", "Printer")
]

import nltk
from nltk.tokenize import word_tokenize

def simple_features(text):
    return {word: True for word in word_tokenize(text.lower())}

featuresets = [(simple_features(text), label) for text, label in sentences]

classifier = nltk.NaiveBayesClassifier.train(featuresets)

# test
test = "My password is not working"
print(classifier.classify(simple_features(test)))

Password


In [2]:
test = "The internet is not working"
print(classifier.classify(simple_features(test)))

Email


In [3]:
test = "The internet is not working"
print(classifier.classify(simple_features(test)))

Email


In [4]:
sentences = [
    ("I forgot my password", "Password"),
    ("My internet is slow", "Connectivity"),
    ("The internet is not working", "Connectivity"),
    ("WiFi is down", "Connectivity"),
    ("Email not working", "Email"),
    ("Cannot send email", "Email"),
    ("Printer is broken", "Printer")
]

In [5]:
featuresets = [(simple_features(text), label) for text, label in sentences]
classifier = nltk.NaiveBayesClassifier.train(featuresets)

test = "The internet is not working"
print(classifier.classify(simple_features(test)))

Connectivity


In [6]:
print(classifier.show_most_informative_features(5))

Most Informative Features
                internet = None            Email : Connec =      2.2 : 1.0
                     not = True            Email : Connec =      2.2 : 1.0
                      my = True           Passwo : Connec =      2.0 : 1.0
                     can = None           Connec : Email  =      1.8 : 1.0
                    send = None           Connec : Email  =      1.8 : 1.0
None


In [7]:
test = "Cannot send my email"
print(classifier.classify(simple_features(test)))


Email
